# Industry Sponsorship as Meta-Evidence — simulations and analysis

An agent-based model of scientific inquiry in which industry-funded Researchers introduce bias
and a central Evaluator aggregates their results under different strategies. The model and the
results reported in the paper are described there; this notebook is the interface to both.

- **Part 1** runs the model. Set the parameters in one dictionary and the notebook translates
  them into the model's arguments.
- **Part 2** analyses the output: correct convergence rates, as a chart and as tables.

**Layout.** Model code is in `code/`, simulation output in `data/`, and this notebook sits in the
root. Paths below are relative to the notebook, so it works from a fresh clone.

In [ ]:
import ast
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CODE_DIR = Path("code") if Path("code").is_dir() else Path(".")
DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")

sys.path.insert(0, str(CODE_DIR.resolve()))  # absolute, so worker processes can import too
from run_2 import BIAS_TYPES, DEFAULT_CONFIG, STRATEGIES, describe_config, run_sweep

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = ":"
plt.rcParams["grid.color"] = "#bbbbbb"
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16

## Part 1 — Running the model

A run sweeps one **bias type** across a range of **bias levels**, under each **Evaluator strategy**.
Every combination is repeated many times, and the convergence rate is the share of repetitions in
which the public agents settle on the best hypothesis.

**Bias types.** Choosing one sets the industry agent class, its inquiry rule, and which parameter
the bias level is written into. The level is always a probability.

| `bias_type` | The bias level is the probability that... |
| --- | --- |
| `design_bias` | an industry study compares its hypothesis against the inferior control |
| `publication_bias` | an industry agent withholds a negative result about its favourite hypothesis |
| `industry_selection` | an industry study of the second-best hypothesis is biased |

**Strategies.** `No adjustment` and `Ignoring industry` are the two benchmarks; `Down-weighting`
and `Relative adjustment` are the two strategies under test. To sweep several down-weighting
weights in one file, write the entry as a pair: `("Down-weighting", 0.1)`.

In [ ]:
# Every configuration key, with the value used when you do not set it.
for key, value in DEFAULT_CONFIG.items():
    print(f"{key:22s} {value}")

### Configure and run

Edit `CONFIG` below; anything left out keeps its default from the table above. These parameters
reproduce the paper's first scenario — small studies, five agents of each kind — on a coarser
grid and with fewer repetitions, so that the notebook runs in a few minutes.

Two parameters interact and are worth stating explicitly. `pulls` is the number of participants
per arm an agent recruits per round, and `study_size_public` / `study_size_industry` is the size
of a single study, so `pulls / study_size` is the number of studies an agent publishes per round.
Setting them equal, as here, gives one study per agent per round.

In [ ]:
CONFIG = {
    "hypotheses": [0.51, 0.5, 0.49],    # success rates, best first
    "control": 0.5,
    "biased_control": 0.48,             # the inferior comparator industry can switch to

    "n_public": 5,
    "n_industry": 5,
    "study_size_public": 200,           # participants per arm, per study
    "study_size_industry": 200,
    "pulls": 200,                       # participants per arm, per agent, per round

    "bias_levels": [0, 0.2, 0.4, 0.6, 0.8, 1],
    "strategies": ["No adjustment", "Ignoring industry",
                   "Down-weighting", "Relative adjustment"],

    "ind_weight": 0.5,                  # weight on industry studies under down-weighting
    "min_studies": 5,                   # studies per source before relative adjustment acts

    "rounds": 100,
    "repetitions": 200,                 # the paper uses 5000
    "seed": None,                       # an int makes the whole sweep reproducible
}

RUN_NAME = "scenario1"
BIAS_TYPES_TO_RUN = ["design_bias", "publication_bias", "industry_selection"]
N_WORKERS = 8                           # None uses all cores but one; 1 runs in this process

In [ ]:
for bias_type in BIAS_TYPES_TO_RUN:
    out_path = DATA_DIR / f"{RUN_NAME}_{bias_type}.csv"
    print(f"\n=== {bias_type} ===")
    run_sweep({**CONFIG, "bias_type": bias_type}, n_workers=N_WORKERS, out_path=out_path)

## Part 2 — Analysis

A run **converges correctly** when the public agents test the best hypothesis, and only it, in
every round of a window at the end of the simulation. That is a demanding criterion: it counts a
late switch onto the best hypothesis as a failure, because it asks for a consensus that is both
correct and stable. The trajectory breakdown at the end of the notebook splits the same window
into the different ways a run can fail.

The functions below read the CSVs written by Part 1. They also read the files used in the paper,
if you place those in `data/`.

In [ ]:
# Columns the analysis reads. Anything else is skipped at read time.
NEEDED_COLUMNS = ["round", "bandits", "pub_bias", "site_bias", "style", "active_hypotheses"]
OPTIONAL_COLUMNS = ["inquiry_ind", "ind_weight"]

STYLE_LABELS = {
    "normal": "No adjustment",
    "ignore": "Ignoring industry",
    "down_weighing_neutral": "Down-weighting",
    "adaptive_adjustment_significance": "Relative adjustment",
    "down_weighing_01": "Down-weighting (w = 0.1)",
    "down_weighing_09": "Down-weighting (w = 0.9)",
}


def parse_set(s):
    """Read an `active_hypotheses` cell back into a set of success rates."""
    if pd.isna(s):
        return set()
    if isinstance(s, set):
        return s

    s = str(s).strip()
    if s == "set()":
        return set()
    if s.startswith("{") and s.endswith("}"):
        inner = s[1:-1]
        return set(float(x.strip()) for x in inner.split(",")) if inner.strip() else set()
    return set()


def map_unique(series, fn):
    """Apply `fn` once per distinct value rather than once per row.

    These columns hold very few distinct strings per file: `bandits` is constant, the bias
    tuples take one value per parameter level, and `active_hypotheses` at most one per
    hypothesis. A row-by-row parse is nearly all waste, and with many hypotheses it is also
    what exhausts memory, because it builds a separate object for every row.
    """
    return series.map({value: fn(value) for value in series.unique()})


def sole_active_from_string(s):
    """The one hypothesis in a raw `active_hypotheses` cell, or NaN if agents are split."""
    active = parse_set(s)
    return next(iter(active)) if len(active) == 1 else np.nan


def series_labels(df):
    """One legend label per row: the strategy, with the weight appended when a file holds
    several down-weighting weights, as in Figure 2 of the paper."""
    labels = df["style"].astype(str).map(lambda s: STYLE_LABELS.get(s, s))

    if "ind_weight" in df.columns:
        down_weighting = df["style"].astype(str) == "down_weighing_neutral"
        if df.loc[down_weighting, "ind_weight"].nunique() > 1:
            weights = df["ind_weight"].map(lambda w: f" (w = {w:g})")
            labels = labels.mask(down_weighting, labels + weights)

    return labels


def load_results(filepath):
    """Read one results CSV and derive the columns the analyses need."""
    available = pd.read_csv(filepath, nrows=0).columns
    missing = [column for column in NEEDED_COLUMNS if column not in available]
    if missing:
        raise ValueError(f"{Path(filepath).name} is missing required column(s): {missing}")

    usecols = [c for c in NEEDED_COLUMNS + OPTIONAL_COLUMNS if c in available]
    df = pd.read_csv(filepath, usecols=usecols)

    # Best hypothesis and the two industry bias levels, parsed once per distinct value.
    df["best_hypothesis"] = map_unique(df["bandits"], lambda s: max(ast.literal_eval(s)))
    df["pub_bias_ind"] = map_unique(df["pub_bias"], lambda s: ast.literal_eval(s)[1])
    df["site_bias_ind"] = map_unique(df["site_bias"], lambda s: ast.literal_eval(s)[1])

    # NaN marks a round in which public agents are split across several hypotheses.
    df["sole_active"] = map_unique(df["active_hypotheses"], sole_active_from_string)
    df["converged_round"] = df["sole_active"] == df["best_hypothesis"]

    df["series"] = series_labels(df)

    # Runs are written consecutively and each one starts at round 1.
    df["unique_run_id"] = (df["round"] == 1).cumsum()

    return df

In [ ]:
SERIES_ORDER = ["Relative adjustment", "Down-weighting", "Ignoring industry", "No adjustment"]


def identify_bias_column(df):
    """Return the column holding the industry bias level that varies, plus a label for it."""
    varies_pub = df["pub_bias_ind"].nunique() > 1 or df["pub_bias_ind"].max() > 0
    varies_site = df["site_bias_ind"].nunique() > 1 or df["site_bias_ind"].max() > 0

    if varies_pub and not varies_site:
        return "pub_bias_ind", "Publication bias"

    # Design bias and industry selection both write into site_bias. They differ in the
    # inquiry rule the industry agents use: the neutral one marks industry selection.
    if "inquiry_ind" in df.columns and df["inquiry_ind"].nunique() == 1:
        label = "Industry selection" if df["inquiry_ind"].iloc[0] == "softmax" else "Design bias"
    else:
        label = "Design bias / Industry selection"

    return "site_bias_ind", label


def summarise_runs(runs, bias_col):
    """Turn one boolean outcome per run into per-(series, bias level) rates with 95% CIs.

    `runs` needs the columns: series, <bias_col>, and 'outcome' (one bool per run).
    """
    rates = runs.groupby(["series", bias_col])["outcome"].agg(["mean", "count", "std"]).reset_index()

    rates["rate"] = rates["mean"] * 100
    rates["ci"] = 1.96 * rates["std"] / np.sqrt(rates["count"]) * 100
    rates["ci"] = rates["ci"].fillna(0)  # a single run has no interval

    return rates


def calculate_convergence_rate(df, n_rounds=45):
    """Share of runs whose public agents test only the best hypothesis throughout the last
    `n_rounds` recorded rounds. Returns (rates, bias_col, bias_name).
    """
    bias_col, bias_name = identify_bias_column(df)

    window = (df.sort_values(["unique_run_id", "round"])
                .groupby("unique_run_id")
                .tail(n_rounds))

    runs = window.groupby("unique_run_id").agg(
        outcome=("converged_round", "all"),
        series=("series", "first"),
        **{bias_col: (bias_col, "first")},
    ).reset_index()

    return summarise_runs(runs, bias_col), bias_col, bias_name


def convergence_table(rates, bias_col, decimals=1):
    """Rates as a wide table: one row per strategy, one column per bias level."""
    cells = rates.assign(cell=[f"{rate:.{decimals}f} +- {ci:.{decimals}f}"
                               for rate, ci in zip(rates["rate"], rates["ci"])])
    table = cells.pivot(index="series", columns=bias_col, values="cell")
    table.insert(0, "n runs", rates.groupby("series")["count"].first())

    table.index.name = "Strategy"
    table.columns.name = "Industry bias"

    ordered = ([s for s in SERIES_ORDER if s in table.index]
               + [s for s in table.index if s not in SERIES_ORDER])
    return table.reindex(ordered)

In [ ]:
SERIES_STYLES = {
    "Relative adjustment": dict(marker="o", color="black", linestyle="-"),
    "Down-weighting": dict(marker="s", color="#333333", linestyle="--"),
    "Ignoring industry": dict(marker="^", color="#666666", linestyle="-."),
    "No adjustment": dict(marker="D", color="#999999", linestyle=":"),
}

# Used for any further series, e.g. several down-weighting weights in one file.
EXTRA_STYLES = [dict(marker="s", color=shade, linestyle="--")
                for shade in ("#111111", "#555555", "#888888", "#bbbbbb")]


def plot_convergence(rates, bias_col, title, ax=None, ylabel="Correct Convergence Rate (%)"):
    """One panel: convergence rate against industry bias, one line per strategy."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    present = set(rates["series"])
    ordered = [s for s in SERIES_ORDER if s in present]
    ordered += sorted(s for s in present if s not in SERIES_ORDER)
    extras = iter(EXTRA_STYLES)

    for name in ordered:
        subset = rates[rates["series"] == name].sort_values(bias_col)
        opts = SERIES_STYLES.get(name) or next(extras, dict(marker="o", color="black", linestyle="-"))
        ax.errorbar(subset[bias_col], subset["rate"], yerr=subset["ci"], label=name,
                    linewidth=1.5, markersize=6, capsize=4, **opts)

    ax.set_xlabel("Industry Bias", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_title(title, fontsize=20, pad=10)
    ax.set_ylim(-5, 105)
    ax.legend(frameon=False, fontsize=16)

    return ax

### Convergence chart and table

`RESULT_FILES` maps a panel title to a results file. Missing files are skipped, so this works
whether you ran one bias type or all three. `WINDOW` is the number of rounds at the end of a run
that must all show correct convergence; the paper uses 45 of 100.

In [ ]:
RESULT_FILES = {
    "Design bias": DATA_DIR / f"{RUN_NAME}_design_bias.csv",
    "Publication bias": DATA_DIR / f"{RUN_NAME}_publication_bias.csv",
    "Industry selection": DATA_DIR / f"{RUN_NAME}_industry_selection.csv",
}
WINDOW = 45
SAVE_FIGURE = True

loaded = {title: load_results(path) for title, path in RESULT_FILES.items() if path.exists()}
missing = [title for title, path in RESULT_FILES.items() if not path.exists()]

if missing:
    print(f"No file for: {', '.join(missing)} - those panels are skipped.")

if loaded:
    fig, axes = plt.subplots(1, len(loaded), figsize=(6 * len(loaded), 6),
                             sharey=True, squeeze=False)

    for ax, (title, df) in zip(axes[0], loaded.items()):
        rates, bias_col, _ = calculate_convergence_rate(df, n_rounds=WINDOW)
        plot_convergence(rates, bias_col, title, ax=ax)
        if ax is not axes[0][0]:
            ax.set_ylabel("")

    plt.tight_layout()

    if SAVE_FIGURE:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{RUN_NAME}_convergence.png", dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
for title, df in loaded.items():
    rates, bias_col, _ = calculate_convergence_rate(df, n_rounds=WINDOW)
    print(f"\n{title} - correct convergence rate (%), with 95% CI")
    display(convergence_table(rates, bias_col))

### Trajectory breakdown

Convergence collapses the end of a run into one bit, which treats every kind of failure alike.
The breakdown splits the same window into mutually exclusive categories that sum to 100%.

Per round the public agents are either all on the best hypothesis (*optimal*), all on the same
worse one (*suboptimal*), or spread across several (*split*). A **switch** is the whole group
moving together. Every switching category is read by comparing where the agents ended the window
with where they started it, so *ended better* means they finished above their starting point, not
that they reached the optimum. *Stable optimal* is the category that tracks the optimum, and it
reproduces the convergence rate above.

| Category | Meaning |
| --- | --- |
| Stable optimal | On the best hypothesis for the whole window |
| Stable suboptimal | On one and the same worse hypothesis throughout |
| One switch (ended better / worse) | A single switch, onto a higher or lower success rate |
| Multiple switches | Two or more switches, optionally split by net direction |
| Never converged | Split in every round of the window |

Split rounds are controlled by `SPLIT_ROUNDS`. With `'ignore'` they are read as transitional and
dropped before switches are detected; with `'separate'`, runs mixing split and converged rounds go
to a further category, **Not fully converged**. Splits need both several public agents and a
stochastic choice, and public agents anneal to `argmax` at the half-way point, so with the paper's
settings the two options give the same tables.

In [ ]:
BASE_TRAJECTORY_CATEGORIES = [
    "Stable optimal",
    "Stable suboptimal",
    "One switch (ended better)",
    "One switch (ended worse)",
]

# Used in place of a single 'Multiple switches' when detail_multiple_switches=True.
MULTIPLE_SWITCH_CATEGORIES = [
    "Multiple switches (ended better)",
    "Multiple switches (ended worse)",
    "Multiple switches (ended same)",
]


def trajectory_categories(split_rounds="ignore", detail_multiple_switches=False):
    """The ordered, exhaustive list of categories a run can fall into."""
    categories = list(BASE_TRAJECTORY_CATEGORIES)
    categories += MULTIPLE_SWITCH_CATEGORIES if detail_multiple_switches else ["Multiple switches"]

    if split_rounds == "separate":
        categories.append("Not fully converged")
    categories.append("Never converged")

    return categories


def classify_trajectory(states, best, split_rounds="ignore", detail_multiple_switches=False):
    """Assign one run's window to a category.

    `states` is the per-round list of sole active hypotheses, NaN where the public agents are
    split; `best` is the optimal hypothesis for that run. Only the endpoints of the path matter,
    so a run that dips and then climbs past where it began still counts as having ended better.
    """
    if all(pd.isna(state) for state in states):
        return "Never converged"

    if split_rounds == "separate" and any(pd.isna(state) for state in states):
        return "Not fully converged"

    # Drop split rounds, then collapse repeats so only genuine switches are left.
    path = []
    for state in states:
        if not pd.isna(state) and (not path or state != path[-1]):
            path.append(state)

    if len(path) == 1:
        return "Stable optimal" if path[0] == best else "Stable suboptimal"

    if len(path) == 2:
        # Consecutive entries always differ, so one switch is either up or down.
        return "One switch (ended better)" if path[1] > path[0] else "One switch (ended worse)"

    if not detail_multiple_switches:
        return "Multiple switches"

    if path[-1] > path[0]:
        return "Multiple switches (ended better)"
    if path[-1] < path[0]:
        return "Multiple switches (ended worse)"
    return "Multiple switches (ended same)"


def calculate_trajectory_breakdown(df, n_rounds=45, split_rounds="ignore",
                                   detail_multiple_switches=False):
    """Place every run in exactly one trajectory category, so that the rates within a
    (series, bias level) cell sum to 100%.

    Returns (breakdown, bias_col, bias_name), with `breakdown` tidy: one row per
    (series, bias level, trajectory).
    """
    if split_rounds not in ("ignore", "separate"):
        raise ValueError(f"split_rounds must be 'ignore' or 'separate', got {split_rounds!r}.")

    bias_col, bias_name = identify_bias_column(df)

    # Only the tail of each run matters, so cut the frame down before any per-row work.
    columns = ["unique_run_id", "round", "sole_active", "best_hypothesis", "series", bias_col]
    window = (df[columns].sort_values(["unique_run_id", "round"])
                         .groupby("unique_run_id").tail(n_rounds))

    runs = window.groupby("unique_run_id").agg(
        states=("sole_active", list),
        best=("best_hypothesis", "first"),
        series=("series", "first"),
        **{bias_col: (bias_col, "first")},
    ).reset_index()

    runs["trajectory"] = [classify_trajectory(states, best, split_rounds, detail_multiple_switches)
                          for states, best in zip(runs["states"], runs["best"])]

    if split_rounds == "ignore":
        affected = np.mean([any(pd.isna(s) for s in states) for states in runs["states"]]) * 100
        print(f"  -> {affected:.1f}% of runs had at least one split round in the last "
              f"{n_rounds} round(s); those rounds are ignored when detecting switches.")

    categories = trajectory_categories(split_rounds, detail_multiple_switches)

    parts = []
    for category in categories:
        runs["outcome"] = runs["trajectory"] == category
        rates = summarise_runs(runs, bias_col)
        rates["trajectory"] = category
        parts.append(rates)

    breakdown = pd.concat(parts, ignore_index=True)
    breakdown["trajectory"] = pd.Categorical(breakdown["trajectory"], categories, ordered=True)
    breakdown = breakdown.sort_values(["series", bias_col, "trajectory"]).reset_index(drop=True)

    return breakdown, bias_col, bias_name


def trajectory_table(breakdown, bias_col, decimals=1):
    """Format a breakdown wide: one row per (series, bias level), one column per category."""
    cells = breakdown.assign(cell=[f"{rate:.{decimals}f} +- {ci:.{decimals}f}"
                                   for rate, ci in zip(breakdown["rate"], breakdown["ci"])])
    table = cells.pivot(index=["series", bias_col], columns="trajectory", values="cell")
    table.insert(0, "n runs",
                 breakdown.groupby(["series", bias_col], observed=True)["count"].first())

    return table

In [ ]:
SPLIT_ROUNDS = "ignore"           # 'ignore' or 'separate'
DETAIL_MULTIPLE_SWITCHES = True   # split 'Multiple switches' by net direction

breakdowns = {}  # title -> (tidy breakdown, bias column), for further work or export

for title, df in loaded.items():
    print(f"\n{title} - last {WINDOW} rounds, percentages with 95% CI")
    breakdown, bias_col, _ = calculate_trajectory_breakdown(
        df, n_rounds=WINDOW, split_rounds=SPLIT_ROUNDS,
        detail_multiple_switches=DETAIL_MULTIPLE_SWITCHES,
    )
    breakdowns[title] = (breakdown, bias_col)
    display(trajectory_table(breakdown, bias_col))